#### Import req Libraries

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

##### Import Utilities and Widgets

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
print(bronze_schema, silver_schema, gold_schema)

In [ ]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-mi/full_load/{data_source}/*.csv'
print(base_path)

### BRONZE

In [ ]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [ ]:
# print check data type
df.printSchema()

In [ ]:
display(df.limit(10))

In [ ]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")